# 🛠️ A Reflective, Self-Correcting Code-Generation Agent

## Learning Objectives
In this notebook, you will learn:
1. **Structured code generation** - A `Code` schema that separates description, imports and code
2. **Execution as the critic** - Run the imports and the code; turn any exception into feedback
3. **Reflect and retry** - Route back to the generator with the error until the code runs or `MAX_ATTEMPTS` is reached
4. **Proving both paths** - A real task, a stub that fails once (the repair path), and a stub that always fails (the cap)

## Where this fits
`03_reflection_architecture.ipynb` showed that a critic which only *reads* code misses bugs and cannot tell whether its fix helped. Here the critique comes from **running** the code: an exception is evidence, not opinion. This is the last notebook of the reflection module; module 12 (`modules/12_advanced_cognitive_patterns/`) continues with advanced cognitive patterns such as self-consistency, tree of thoughts and tree search.

## Prerequisites
- `03_reflection_architecture.ipynb` (structured generate → critique → refine)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

> **Note**: this agent runs model-written code with `exec` on your machine. Keep the tasks self-contained (no network, no credentials) and use a sandbox - e.g. the hosted interpreter in `modules/02_tools_and_function_calling/notebooks/06_hosted_code_execution.ipynb` - for anything untrusted.

---
## 📖 Part 1: The Self-Correcting Loop

![](https://i.imgur.com/koLDCZL.png)

This project focuses on building a **Reflective Self-Correcting Code Generation AI Agent**, designed to iteratively generate, execute, and refine code to achieve accurate solutions. The workflow integrates reflective reasoning and error analysis to ensure robust and functional code generation. The workflow includes the following components:

1. **Code Generation**:
   - The system uses `gpt-4o-mini` to generate code based on the user's input and problem requirements.
   - Code is generated following a predefined schema, including:
     - **Prefix**: Problem description or setup requirements.
     - **Imports**: Necessary libraries or dependencies.
     - **Code**: Functional implementation of the solution.

2. **Code Execution and Reflection**:
   - The generated code is executed, and the results are analyzed.
   - If an error occurs, feedback is provided to refine the code:
     - **Error Feedback**: Captures errors or unexpected behaviors during execution.
     - **Reflection**: the model reflects on the error feedback to iteratively improve the solution.

3. **Iterative Correction Loop**:
   - The system repeats the generate-execute-reflect cycle until:
     - A successful solution is achieved (no errors).
     - The attempt count exceeds a predefined threshold (`N`), ensuring the loop terminates if a solution cannot be reached.

4. **Attempt Counter**:
   - Tracks the number of attempts (`K`) and increments with each iteration.
   - If the attempt count exceeds the predefined threshold, the process halts, and the user is informed.

5. **Final Validation**:
   - Once the code passes execution without errors, the refined solution is presented to the user as the final output.

---
## 📦 Step 1: Environment Setup

Load the API key and create the chat model through the course factory.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: The code generator
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🧱 Part 2: Structured Code Generation

The generator returns a `Code` object with three fields - a **prefix** describing the approach, the **imports**, and the **code** - so the checker can run the imports on their own first. The system prompt also asks for code that *runs something*, so executing it is a real test.

In [ ]:
# ============================================================================
# CODE GENERATOR: System prompt + Code schema + structured output
# ============================================================================
from pydantic import BaseModel, Field


class Code(BaseModel):
    """Schema for code solutions to questions about coding."""

    prefix: str = Field(description="Description of the problem and approach")
    imports: str = Field(description="Just the import statements of the code")
    code: str = Field(description="Code blocks not including import statements")


CODE_GEN_SYS_PROMPT = [
        (
            "system",
            """You are a coding assistant.
                Ensure any code you provide can be executed with all required imports and variables defined.
                Make sure point 3 below has some code to run and execute any code or functions which you define

                Structure your answer as follows:
                  1) a prefix describing the code solution
                  2) the imports (if no imports needed keep it empty string)
                  3) the functioning code blocks

                Here is the user question:""",
        )
]

code_generator = llm.with_structured_output(Code)

result = code_generator.invoke(CODE_GEN_SYS_PROMPT + [("user", "How to create fibonacci series")])
print(f"Prefix: {result.prefix}\n\nImports: {result.imports or '(none)'}\n\nCode:\n{result.code}")

---
## ⚙️ Part 3: Generate, Check, Decide

### 3.1 State and the Generate Node

The state keeps the conversation (`messages`), the latest `code_solution`, an `attempts` counter and an `error_flag`. Nodes return **only the new messages** - the `add_messages` reducer appends them. (The original version appended to the state's list in place *and* returned it, which duplicated the history on every step.)

The generate node is built around a **swappable generator**, so Part 5 can plug in stubs.

In [ ]:
# ============================================================================
# STATE + GENERATE NODE (built around any runnable that returns Code)
# ============================================================================
from typing import Annotated, List, TypedDict

from langgraph.graph.message import AnyMessage, add_messages


class CodeGenState(TypedDict):
    """
    error_flag : "yes" if the last check failed, else "no"
    messages : user question, solution attempts, error feedback
    code_solution : latest Code object
    attempts : number of generations so far
    """
    error_flag: str
    messages: Annotated[List[AnyMessage], add_messages]
    code_solution: Code
    attempts: int


def make_generate_node(generator):
    def generate_code(state: CodeGenState) -> dict:
        """Generate a code solution (structured as prefix / imports / code)."""
        attempt = state["attempts"] + 1
        print(f"--- ✍️  GENERATING CODE SOLUTION (attempt {attempt}) ---")
        code_soln = generator.invoke(CODE_GEN_SYS_PROMPT + state["messages"])
        summary = (f"Here is my solution attempt:\n\nDescription: {code_soln.prefix}\n\n"
                   f"Imports: {code_soln.imports}\n\nCode:\n{code_soln.code}")
        return {"messages": [("assistant", summary)], "code_solution": code_soln, "attempts": attempt}

    return generate_code


print("✅ State and generate node factory defined")

### 3.2 The Check Node: Execution as the Critic

`exec` the imports, then the imports + code. Any exception becomes a user message that asks the model to **reflect** on what went wrong and return a full new solution.

In [ ]:
# ============================================================================
# CHECK NODE: Run imports, then the code; an exception becomes feedback
# ============================================================================
import contextlib
import io


def check_code_execution(state: CodeGenState) -> dict:
    print("--- 🧪 CHECKING CODE EXECUTION ---")
    code_soln = state["code_solution"]

    try:
        exec(code_soln.imports, {})
    except Exception as e:
        print(f"--- ❌ IMPORT CHECK FAILED: {type(e).__name__}: {e} ---")
        feedback = f"Import test failed!\nHere is the exception trace details:\n{e}.\n\nPlease fix the import section."
        return {"messages": [("user", feedback)], "error_flag": "yes"}

    captured = io.StringIO()
    try:
        with contextlib.redirect_stdout(captured):  # keep the program's own output out of the log
            exec(f"{code_soln.imports}\n{code_soln.code}", {})
    except Exception as e:
        print(f"--- ❌ CODE BLOCK CHECK FAILED: {type(e).__name__}: {e} ---")
        feedback = f"""Your code solution failed the code execution test!
Here is the exception trace details:
{type(e).__name__}: {e}

Reflect on this error and your prior attempt to solve the problem.

(1) State what you think went wrong with the prior solution
(2) try to solve this problem again.

Return the FULL SOLUTION.

Use the code tool to structure the output with a prefix, imports, and code block."""
        return {"messages": [("user", feedback)], "error_flag": "yes"}

    print(f"--- ✅ NO ERRORS FOUND ({len(captured.getvalue().splitlines())} lines of program output) ---")
    return {"error_flag": "no"}


print("✅ check_code_execution defined")

### 3.3 Conditional Routing

A pure router: finish when the code ran, or when `MAX_ATTEMPTS` generations have been used; otherwise go back to the generator with the error in the conversation.

In [ ]:
# ============================================================================
# ROUTER: finish on success or at the attempt cap, else retry
# ============================================================================
MAX_ATTEMPTS = 3


def decide_next(state: CodeGenState) -> str:
    if state["error_flag"] == "no" or state["attempts"] >= MAX_ATTEMPTS:
        print("--- 🏁 DECISION: FINISH ---")
        return "__end__"
    print("--- 🔁 DECISION: RETRY ---")
    return "generate_code"


print(f"✅ Router ready (MAX_ATTEMPTS={MAX_ATTEMPTS})")

---
## 🏗️ Part 4: The Agent Graph

`generate_code → check_code → (generate_code | END)`. `build_coder_agent` takes the generator as an argument.

In [ ]:
# ============================================================================
# GRAPH: generate_code -> check_code -> (generate_code | END)
# ============================================================================
from langgraph.graph import END, START, StateGraph


def build_coder_agent(generator):
    graph = StateGraph(CodeGenState)
    graph.add_node("generate_code", make_generate_node(generator))
    graph.add_node("check_code", check_code_execution)
    graph.add_edge(START, "generate_code")
    graph.add_edge("generate_code", "check_code")
    graph.add_conditional_edges("check_code", decide_next, [END, "generate_code"])
    return graph.compile()


def call_reflection_coding_agent(agent, prompt: str) -> dict:
    final = agent.invoke({"messages": [("user", prompt)], "attempts": 0, "error_flag": "no"})
    soln = final["code_solution"]
    status = "runs ✅" if final["error_flag"] == "no" else "still failing ❌"
    print(f"\n📊 {final['attempts']} attempt(s), final code {status}")
    print(f"\nDescription:\n{soln.prefix}\n\nCode:\n{soln.imports}\n\n{soln.code}")
    return final


coder_agent = build_coder_agent(code_generator)
print("🧩 Nodes:", [n for n in coder_agent.get_graph().nodes if not n.startswith("__")])

---
## 🧪 Part 5: Proving Both Paths

### 5.1 ✅ A Real Task

A self-contained task that `gpt-4o-mini` usually solves on the first attempt - the loop's happy path.

In [ ]:
# ============================================================================
# RUN 1: A real task (usually passes on the first attempt)
# ============================================================================
final_pivot = call_reflection_coding_agent(coder_agent, "write some code to demonstrate how to do a pivot table in pandas")

### 5.2 🔁 A Failing First Attempt (the Repair Path)

The repair path is the point of the agent, but a good model rarely fails on demand. `FlakyGenerator` returns a solution with a real bug - it uses `statistics` without importing it - on its **first** call, and asks the real model after that. The model then sees the actual `NameError` and must repair the code.

In [ ]:
# ============================================================================
# STUB: The first solution has a missing import; later calls use the real model
# ============================================================================
class FlakyGenerator:
    def __init__(self, real_generator):
        self.real_generator = real_generator
        self.calls = 0

    def invoke(self, messages):
        self.calls += 1
        if self.calls == 1:
            print("   🎭 STUB: returning code with a missing import")
            return Code(
                prefix="Compute the mean and median of a list of numbers.",
                imports="",
                code="data = [3, 1, 4, 1, 5, 9, 2, 6]\nprint(statistics.mean(data), statistics.median(data))",
            )
        return self.real_generator.invoke(messages)


final_repaired = call_reflection_coding_agent(
    build_coder_agent(FlakyGenerator(code_generator)),
    "write code that prints the mean and median of the list [3, 1, 4, 1, 5, 9, 2, 6]",
)
assert final_repaired["attempts"] >= 2 and final_repaired["error_flag"] == "no"
print("\n✅ Repaired after seeing the error")

### 5.3 🛑 A Generator That Never Gets It Right (the Cap)

A stub that always returns failing code - no LLM calls at all - proves that `MAX_ATTEMPTS` ends the loop and that the agent reports the code as still failing rather than pretending it works.

In [ ]:
# ============================================================================
# STUB: Always-broken code - the attempt cap must stop the loop
# ============================================================================
class BrokenGenerator:
    def invoke(self, messages):
        return Code(prefix="Always broken.", imports="", code="raise RuntimeError('still broken')")


final_capped = call_reflection_coding_agent(build_coder_agent(BrokenGenerator()), "anything")
assert final_capped["attempts"] == MAX_ATTEMPTS and final_capped["error_flag"] == "yes"
print(f"\n✅ Stopped at the cap ({MAX_ATTEMPTS} attempts)")

To go further, try tasks that depend on the outside world - *"scrape data from a Wikipedia page"* or *"sentiment analysis using transformers"*. Expect the loop to hit its cap more often: a missing API key or a model download is not something the generator can fix by rewriting code, and those runs call the network from inside `exec`.

---
## 📝 Summary

In this notebook, we learned:

### 1. Structured code generation
- A `Code` schema separates description, imports and code, so the checker can test the imports first.

### 2. Execution as the critic
- An exception from `exec` is evidence, not opinion - the feedback message carries the actual error to the next attempt.

### 3. Reflect and retry
- Nodes return only new messages; a pure router ends on success or at `MAX_ATTEMPTS`.

### 4. Proving both paths
- A swappable generator lets stubs show the repair path (one failure, then a fix) and the cap (never fixed) deterministically.
- "It runs" is a weak test: add assertions or hidden tests (as in `03_reflection_architecture.ipynb`) when correctness matters.

### Next Steps
- **`modules/12_advanced_cognitive_patterns/`**: advanced cognitive patterns - self-consistency, tree of thoughts, LATS, simulation and metacognition